# Index Analysis

In notebook **01_discovery** we spotted several suspicious indexes — unused PKs on time-series tables, a redundant email index, and some large composite indexes with zero scans. Here we use the `Forensic` class to systematically classify every index and get a health score.

The forensic workflow:
1. Fetch index metadata from `pg_stat_user_indexes` and selectivity from `pg_stats`
2. Classify each index as `CRITICAL`, `SUSPICIOUS`, `HEALTHY`, `PK`, or `UNIQUE`
3. Compute a 0–100 health score per index
4. Produce a ranked list of drop candidates

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config SqlMagic.displaylimit = None
%load_ext sql

## Classification Guide

The `Forensic` class uses these rules:

| Status | Condition | Action |
|---|---|---|
| 🔴 **CRITICAL** | `idx_scan = 0` + large (＞ 40% of table) | Drop candidate — pure waste |
| 🟠 **SUSPICIOUS** | `idx_scan < 100` + not PK/unique | Needs review — low value |
| 🟢 **HEALTHY** | Actively used | Keep — justifies its cost |
| 🔵 **PK** | Primary key index | Always keep — integrity |
| 🔷 **UNIQUE** | Unique constraint | Keep — integrity (even if usage is moderate) |

**Key columns:**
- `idx_scan` — how many times the index was actually used
- `pct_of_table` — index size relative to table heap
- `selectivity` — ratio of distinct values to total rows (closer to 0 = more selective = more useful)

## 1. Raw Index Overview

Every index sorted by size — lets us quickly spot the biggest storage consumers.

In [2]:
%%sql
SELECT 
    s.schemaname,
    s.relname AS table_name,
    s.indexrelname AS index_name,
    pg_size_pretty(pg_relation_size(s.indexrelid)) AS index_size,
    pg_size_pretty(pg_relation_size(t.relid)) AS table_size,
    s.idx_scan,
    ROUND(
        pg_relation_size(s.indexrelid)::numeric / 
        NULLIF(pg_relation_size(t.relid), 0) * 100, 
        2
    ) AS pct_of_table
FROM pg_stat_user_indexes s
JOIN pg_stat_user_tables t 
    ON s.relid = t.relid
ORDER BY pg_relation_size(s.indexrelid) DESC;

The top offenders are already visible: `idx_metric_time` (481 MB, 0 scans), `events_pkey` (391 MB, 0 scans), `idx_session` (85 MB, 0 scans).

## 2. Forensic Classification

The `Forensic.check_indexes()` method enriches the raw data with selectivity, classifies every index, and computes a score.

In [3]:
import os
from database_as_crime_scene.forensic.forensic import Forensic

database_url = os.getenv('DATABASE_URL', 'localhost:5432')
forensic = Forensic(database_url)
df = forensic.check_indexes()
df

### Reading the forensic report

| Column | What it tells you |
|---|---|
| `status` | Classification: CRITICAL / SUSPICIOUS / HEALTHY / PK / UNIQUE |
| `score` | 0–100 usefulness score (higher = better). Below 50 = consider dropping |
| `idx_scan` | How many times the index was used (since last stats reset) |
| `selectivity` | Low (close to 0) = high cardinality = index is useful for lookups. High (close to 1) = low cardinality = index is less effective |
| `pct_of_table` / `pct_vs_heap` | Index size vs data size — overhead gauge |
| `is_pk` / `is_unique` | Never drop these without understanding the integrity constraint |

## 3. Findings Summary

### 🔴 CRITICAL (drop candidates)

| Index | Size | `idx_scan` | Selectivity | Why |
|---|---|---|---|---|
| `idx_users_profile_email` | 0.55 MB | 0 | 1.0 (unique) | **Duplicate** — `users_profile_email_key` already covers this. Drop the redundant one. |
| `idx_metric_time` | 481 MB | 0 | 0.0 (metric_name) / 1.0 (recorded_at) | Composite index on `(metric_name, recorded_at)`. metric_name has only 5 distinct values (low cardinality = terrible for btree). Zero usage. |

### 🟠 SUSPICIOUS (needs review)

| Index | Size | `idx_scan` | Selectivity | Why |
|---|---|---|---|---|
| `idx_friends_user_id` | 0.14 MB | 0 | 0.38 | Friends PK already covers `user_id` as leading column. Likely redundant. |
| `idx_friends_friend_id` | 0.17 MB | 0 | 0.62 | FK index on friend_id — never used. May be needed for cascade operations. |

### 🟢 HEALTHY (keep)

All social table indexes with actual usage (`posts_pkey`, `users_profile_pkey`, `friends_pkey`, `users_profile_email_key`) are fine. The FK indexes on `comments` and `posts` with low-or-zero scans but small size are marked HEALTHY because they're small relative to their tables — the cost of keeping them is negligible compared to the risk of missing a future query pattern.

### 🔵 PK — 0 scans but kept by rule

These PK indexes show `idx_scan = 0` but are protected by the `is_pk` flag:
- `events_pkey` — 391 MB
- `metrics_pkey` — 214 MB
- `logs_pkey` — 214 MB
- `comments_pkey` — 26 MB

They exist on UUID/serial columns that are never searched by PK. While we can't drop them (they're the table's identity), their size is pure overhead for insert-heavy workloads.

## 4. Action Plan

| Priority | Action | SQL | Saves |
|---|---|---|---|
| P0 | Drop redundant `idx_users_profile_email` (duplicates unique constraint) | `DROP INDEX IF EXISTS idx_users_profile_email;` | 0.55 MB + faster writes |
| P1 | Investigate `idx_metric_time` — replace with partial index or drop | Requires understanding query patterns first | 481 MB + write overhead |
| P2 | Investigate `idx_session` on logs — preserved only if session-based queries exist | `DROP INDEX IF EXISTS idx_session;` if unused | 85 MB + write overhead |
| P3 | Review `idx_friends_user_id` — likely redundant given PK on `(user_id, friend_id)` | Validate then `DROP INDEX IF EXISTS idx_friends_user_id;` | 0.14 MB |
| P4 | Consider partitioning time-series tables (notebook **06**) to shrink PK index sizes per partition | Range partition by time | ~800 MB in PK overhead for active partition only |

**⚠️ Always verify an index is unused before dropping** — `pg_stat_user_indexes` resets on server restart, so 0 scans may mean "reset since last use." Run a workload first, then re-check.